## Example

https://endjin.com/blog/a-look-into-pandera-and-great-expectations-for-data-validation

In [2]:
import pandas as pd
import pandera.pandas as pa

# data to validate
df = pd.DataFrame({
    "column1": [1, 2, 3],
    "column2": [1.1, 1.2, 1.3],
    "column3": ["a", "b", "c"],
})

schema = pa.DataFrameSchema({
    "column1": pa.Column(int, pa.Check.ge(0)),
    "column2": pa.Column(float, pa.Check.lt(10)),
    "column3": pa.Column(
        str,
        [
            pa.Check.isin([*"abc"]),
            pa.Check(lambda series: series.str.len() == 1),
        ]
    ),
}
)

validated_df = schema.validate(df)
print(validated_df)

   column1  column2 column3
0        1      1.1       a
1        2      1.2       b
2        3      1.3       c


In [6]:
# define a schema
class Schema(pa.DataFrameModel):
    column1: int = pa.Field(ge=0)
    column2: float = pa.Field(lt=10)
    column3: str = pa.Field(isin=[*"abc"])

    @pa.check("column3")
    def custom_check(cls, series: pd.Series) -> pd.Series:
        return series.str.len() == 2
try:
    Schema.validate(df)
except (Exception, BaseException) as e:
    import traceback
    tb = traceback.TracebackException.from_exception(e)
    print(tb)

Column 'column3' failed element-wise validator number 1: <Check custom_check> failure cases: a, b, c


## Real apply

In [7]:
import pandas as pd 
import numpy as np

df_all = pd.read_excel("/home/user/py-pannguyen-learn/data/raw/sales_data_sample.xlsx", sheet_name=None)
df_all.keys()

dict_keys(['Sales_Data', 'Customer_Data', 'Inventory', 'Monthly_Summary', 'Data_Issues'])

In [8]:
df_sales_data = pd.DataFrame(df_all["Sales_Data"])
df_sales_data.head()

,Order_ID,Date,Category,Region,Customer_Segment,Payment_Method,Quantity,Unit_Price,Discount_Rate,Customer_Rating,Returned,Subcategory,Total_Price,Discount_Amount,Net_Sales,Shipping_Cost,Profit_Margin,Profit,Month
0,ORD-10000,2024-07-12,Books,East,Business,PayPal,1,316.58,0.20,5.0,False,Comics,316.58,63.3160,253.2640,0.0,0.33,83.577120,2024-07
1,ORD-10001,2025-03-15,Sports,West,Premium,Bank Transfer,5,279.76,0.10,NaN,True,Water Sports,1398.80,139.8800,1258.9200,0.0,0.20,251.784000,2025-03
2,ORD-10002,2024-12-27,Home & Kitchen,Central,Business,Bank Transfer,6,209.61,0.05,4.0,False,Furniture,1257.66,62.8830,1194.7770,0.0,0.37,442.067490,2024-12
3,ORD-10003,2024-07-16,Home & Kitchen,North,Regular,Credit Card,3,265.47,0.15,4.0,False,Kitchenware,796.41,119.4615,676.9485,0.0,0.23,155.698155,2024-07
4,ORD-10004,2024-06-11,Electronics,North,New,Bank Transfer,9,449.32,0.00,4.0,False,Accessories,4043.88,0.0000,4043.8800,0.0,0.37,1496.235600,2024-06


In [9]:
df_sales_data.columns

Index(['Order_ID', 'Date', 'Category', 'Region', 'Customer_Segment',
       'Payment_Method', 'Quantity', 'Unit_Price', 'Discount_Rate',
       'Customer_Rating', 'Returned', 'Subcategory', 'Total_Price',
       'Discount_Amount', 'Net_Sales', 'Shipping_Cost', 'Profit_Margin',
       'Profit', 'Month'],
      dtype='object')

In [12]:
import pandera.pandas as pa
import pandas as pd

df_sales_data_validation = pa.DataFrameSchema({
    "Order_ID": pa.Column(str, required=True, nullable=True),

    "Date": pa.Column(
        str,
        checks=[
            # --- Check 1: correct datetime format ---
            pa.Check(
                lambda x: pd.to_datetime(
                    x,
                    format="%Y-%m-%d %H:%M:%S",
                    errors="coerce"
                ),
                element_wise=True,
                error="Incorrect format. Use YYYY-MM-DD HH:MM:SS"
            ),

            # --- Check 2: range validation ---
            pa.Check(
                lambda x: (
                   pd.to_datetime("2024-01-01 00:00:00") <= pd.to_datetime(x, errors="coerce") <= pd.to_datetime("2025-01-01 00:00:00")
                ),
                element_wise=True,
                error="Date not in allowed range: 2024"
            )
        ],
        coerce=True,
        nullable=True,
        required=True,
    )
})


try:
    df_sales_data_validation.validate(df_sales_data)
except Exception as e:
    tb = traceback.TracebackException.from_exception(e)
    print(tb)


/home/user/py-pannguyen-learn/.venv/lib/python3.11/site-packages/pandera/backends/pandas/checks.py:239: FutureWarning: 'all' with datetime64 dtypes is deprecated and will raise in a future version. Use (obj != pd.Timestamp(0)).all() instead.
  check_output.all(),
/home/user/py-pannguyen-learn/.venv/lib/python3.11/site-packages/pandera/backends/pandas/checks.py:202: FutureWarning: 'all' with datetime64 dtypes is deprecated and will raise in a future version. Use (obj != pd.Timestamp(0)).all() instead.
  if check_output.all():


Column 'Date' failed element-wise validator number 1: <Check <lambda>: Date not in allowed range: 2024> failure cases: 2025-03-15 00:00:00, 2025-02-25 00:00:00, 2025-03-26 00:00:00, 2025-02-03 00:00:00, 2025-03-10 00:00:00, 2025-01-19 00:00:00, 2025-01-02 00:00:00, 2025-02-08 00:00:00, 2025-03-11 00:00:00, 2025-03-30 00:00:00, 2025-02-14 00:00:00, 2025-02-01 00:00:00, 2025-02-23 00:00:00, 2025-02-10 00:00:00, 2025-03-12 00:00:00, 2025-03-06 00:00:00, 2025-03-26 00:00:00, 2025-02-04 00:00:00, 2025-03-17 00:00:00, 2025-01-29 00:00:00, 2025-01-21 00:00:00, 2025-03-04 00:00:00, 2025-01-05 00:00:00, 2025-01-26 00:00:00, 2025-03-11 00:00:00, 2025-02-21 00:00:00, 2025-03-10 00:00:00, 2025-01-14 00:00:00, 2025-02-22 00:00:00, 2025-01-18 00:00:00, 2025-03-26 00:00:00, 2025-03-25 00:00:00, 2025-01-08 00:00:00, 2025-03-12 00:00:00, 2025-03-18 00:00:00, 2025-02-21 00:00:00, 2025-01-09 00:00:00, 2025-01-09 00:00:00, 2025-03-06 00:00:00, 2025-01-11 00:00:00, 2025-02-25 00:00:00, 2025-03-14 00:00:00,